In [5]:
class BankNode:
    def __init__(self, name):
        self.name = name
        self.ledger = []       # The Paxos Log
        self.balance = 0       # The current State
        self.promised_id = -1

    def apply_transaction(self, tx):
        """The 'State Machine' logic: Executes the agreed command"""
        op, amount = tx
        if op == "CREDIT":
            self.balance += amount
        elif op == "DEBIT":
            self.balance -= amount
        self.ledger.append(tx)

    def handle_accept(self, prop_id, tx, index):
        """Paxos Phase 2: Accept the transaction at a specific index"""
        if prop_id >= self.promised_id:
            # In a real system, we'd check if index == len(self.ledger)
            self.apply_transaction(tx)
            return True
        return False

# --- THE SIMULATION ---
cluster = [BankNode("Server-1"), BankNode("Server-2"), BankNode("Server-3")]
leader_proposal_id = 500  # Previously established via Phase 1

# Standard Financial Transactions
transactions = [
    ("CREDIT", 1000), # Deposit
    ("DEBIT", 200),   # Withdrawal
    ("CREDIT", 50),   # Interest
    ("DEBIT", 150)    # Payment
]

print(f"{'Index':<8} | {'Command':<12} | {'Status':<10}")
print("-" * 35)

for i, tx in enumerate(transactions):
    # Leader sends the 'Accept' request to the cluster
    success_count = sum([node.handle_accept(leader_proposal_id, tx, i) for node in cluster])
    
    status = "COMMITTED" if success_count > len(cluster)//2 else "FAILED"
    print(f"{i:<8} | {str(tx):<12} | {status:<10}")

# Final "Proof of Consistency"
print("\n--- Final State Check ---")
for node in cluster:
    print(f"{node.name} Balance: ${node.balance} (Log Length: {len(node.ledger)})")

Index    | Command      | Status    
-----------------------------------
0        | ('CREDIT', 1000) | COMMITTED 
1        | ('DEBIT', 200) | COMMITTED 
2        | ('CREDIT', 50) | COMMITTED 
3        | ('DEBIT', 150) | COMMITTED 

--- Final State Check ---
Server-1 Balance: $700 (Log Length: 4)
Server-2 Balance: $700 (Log Length: 4)
Server-3 Balance: $700 (Log Length: 4)
